Explore hyper-parameters

Training YOLOv8n with Real Waste images in Colab.
This will capture training time.

In [25]:
import os
print(os.getcwd())

/Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml


Because Colab uses temporarly environments, we need to install ultralytics every time I need to downlaod a model and train it.

In [26]:
#skipping this cell bc i am running locally on my mac so i can skip the content code

#import os
#print(os.listdir('/content'))

Mount google drive in this environment

In [27]:
# I am running this locally on my Mac so I can skip these lines

#from google.colab import drive
#drive.mount('/content/drive')

In [28]:
!pip install ultralytics

Check if we have GPU working.

In [29]:
import torch
print(torch.cuda.is_available())
# On Mac, CUDA usually returns False. That is okay because training can use MPS.
# Should return: True

False


In [30]:
#test to make sure input and output directories exist

from pathlib import Path

SOURCE_DIR = Path("data/realwaste-main/RealWaste")

print("SOURCE_DIR exists:", SOURCE_DIR.exists())
print("Folders inside SOURCE_DIR:")

if SOURCE_DIR.exists():
    for item in SOURCE_DIR.iterdir():
        if item.is_dir():
            print(item.name)

SOURCE_DIR exists: True
Folders inside SOURCE_DIR:
Paper
Metal
Cardboard
Food Organics
Glass
Vegetation
Textile Trash
Miscellaneous Trash
Plastic


In [31]:
from pathlib import Path

DATASET_DIR = Path("/data/realwaste-main/RealWaste")

print("Dataset exists:", DATASET_DIR.exists())

for split in ["train", "val", "test"]:
    split_dir = DATASET_DIR / split

    print(f"\n{split.upper()} exists:", split_dir.exists())

    if split_dir.exists():
        print("Classes:")
        for item in sorted(split_dir.iterdir()):
            if item.is_dir():
                print(item.name)

Dataset exists: False

TRAIN exists: False

VAL exists: False

TEST exists: False


Create stratified YOLO classification dataset

Training Block

In [32]:
#add a new automatic random split as a baseline to compare against other stratified splits

from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from collections import Counter

SOURCE_DIR = Path("data/realwaste-main/RealWaste")
OUTPUT_DIR = Path("outputs_for_YOLO_classification_baseline")

train_ratio = 0.70
seed = 42

image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "val", "test"]:
    (OUTPUT_DIR / split).mkdir(parents=True, exist_ok=True)

image_paths = []
targets = []

class_names = sorted([p.name for p in SOURCE_DIR.iterdir() if p.is_dir()])

for class_name in class_names:
    class_folder = SOURCE_DIR / class_name
    for img_path in class_folder.iterdir():
        if img_path.suffix.lower() in image_extensions:
            image_paths.append(img_path)
            targets.append(class_name)

print("Classes found:", class_names)
print("Total images:", len(image_paths))
print("Original class counts:", Counter(targets))

train_paths, temp_paths, train_targets, temp_targets = train_test_split(
    image_paths,
    targets,
    train_size=train_ratio,
    random_state=seed
)

val_paths, test_paths, val_targets, test_targets = train_test_split(
    temp_paths,
    temp_targets,
    test_size=0.5,
    random_state=seed
)

def copy_split(paths, labels, split_name):
    for src_path, class_name in zip(paths, labels):
        dest_dir = OUTPUT_DIR / split_name / class_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dest_dir / src_path.name)

copy_split(train_paths, train_targets, "train")
copy_split(val_paths, val_targets, "val")
copy_split(test_paths, test_targets, "test")

print("Done creating RANDOM baseline YOLO classification folders!")
print("Train:", len(train_paths), Counter(train_targets))
print("Val:", len(val_paths), Counter(val_targets))
print("Test:", len(test_paths), Counter(test_targets))

Classes found: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
Total images: 4752
Original class counts: Counter({'Plastic': 921, 'Metal': 790, 'Paper': 500, 'Miscellaneous Trash': 495, 'Cardboard': 461, 'Vegetation': 436, 'Glass': 420, 'Food Organics': 411, 'Textile Trash': 318})
Done creating RANDOM baseline YOLO classification folders!
Train: 3326 Counter({'Plastic': 656, 'Metal': 567, 'Paper': 351, 'Miscellaneous Trash': 351, 'Cardboard': 313, 'Vegetation': 295, 'Glass': 288, 'Food Organics': 283, 'Textile Trash': 222})
Val: 713 Counter({'Plastic': 135, 'Metal': 105, 'Glass': 76, 'Food Organics': 72, 'Miscellaneous Trash': 72, 'Cardboard': 70, 'Paper': 68, 'Vegetation': 65, 'Textile Trash': 50})
Test: 713 Counter({'Plastic': 130, 'Metal': 118, 'Paper': 81, 'Cardboard': 78, 'Vegetation': 76, 'Miscellaneous Trash': 72, 'Glass': 56, 'Food Organics': 56, 'Textile Trash': 46})


In [33]:
#verify output directory exists

from pathlib import Path

OUTPUT_DIR = Path("outputs_for_YOLO_classification_baseline")

for split in ["train", "val", "test"]:
    split_dir = OUTPUT_DIR / split
    print(f"\n{split.upper()} exists:", split_dir.exists())

    class_counts = {}
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*")))
            class_counts[class_dir.name] = count

    print(class_counts)


TRAIN exists: True
{'Cardboard': 313, 'Food Organics': 283, 'Glass': 288, 'Metal': 567, 'Miscellaneous Trash': 351, 'Paper': 351, 'Plastic': 656, 'Textile Trash': 222, 'Vegetation': 295}

VAL exists: True
{'Cardboard': 70, 'Food Organics': 72, 'Glass': 76, 'Metal': 105, 'Miscellaneous Trash': 72, 'Paper': 68, 'Plastic': 135, 'Textile Trash': 50, 'Vegetation': 65}

TEST exists: True
{'Cardboard': 78, 'Food Organics': 56, 'Glass': 56, 'Metal': 118, 'Miscellaneous Trash': 72, 'Paper': 81, 'Plastic': 130, 'Textile Trash': 46, 'Vegetation': 76}


In [34]:

from ultralytics import YOLO
import torch
from datetime import datetime

# Start time
start_time = datetime.now()
print("Training started at:", start_time.strftime("%Y-%m-%d %H:%M:%S"))
# ======> Train the model
def main():

    # 1. Check for NVIDIA GPU (Colab/PC)
    # 2. Check for Apple Silicon (Mac)
    # 3. Default to CPU
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

    print(f"I am training on: {device.upper()}")

    ### Smaller model.
    model = YOLO("yolov8n-cls.pt")

    model.train(
        ### data="realwaste-yolo/RealWaste/data.yaml",
        ### '/content/RealWaste/data.yaml'
        data="outputs_for_YOLO_classification_baseline",
        epochs=10, ###==> Hyper-parameters
        lr0=0.005,
        lrf=0.1,
        momentum=0.9,
        weight_decay=0.0001,
        ###warmup_epochs=5,
        ###warmup_momentum=0.8,
        ###warmup_bias_lr=0.1, ###<==
        imgsz=224,
        batch=16,
        patience=25,
        device=device,
        workers=0,  # safer in Jupyter if needed
        project="runs",
        name="yolov8n_cls_baseline_10epochs"
    )

if __name__ == "__main__":
    main()
# <=====

# End time
end_time = datetime.now()
print("Training ended at:", end_time.strftime("%Y-%m-%d %H:%M:%S"))

# Elapsed time
elapsed = end_time - start_time

# Format nicely (hours, minutes, seconds)
hours, remainder = divmod(elapsed.total_seconds(), 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Elapsed time: {int(hours)}h {int(minutes)}m {int(seconds)}s")

Training started at: 2026-05-05 22:19:43
I am training on: MPS
Ultralytics 8.4.46 🚀 Python-3.11.15 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=outputs_for_YOLO_classification_baseline, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=yolov8n_cls_baseline_10epochs-4, nbs=64, n

In [35]:
#code to actually test the model, and see what the test accuracy is
#very important was missing before but we need test accuracy after each run to verify whether or not the model is improving or not

from ultralytics import YOLO

# Load best trained model
model = YOLO("runs/classify/runs/yolov8n_cls_baseline_10epochs/weights/best.pt")

# Evaluate on test dataset
metrics = model.val(
    data="outputs_for_YOLO_classification_baseline",
    split="test"
)

print("\nEvaluation Metrics:")
print(metrics)

print(f"\nTop-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

Ultralytics 8.4.46 🚀 Python-3.11.15 torch-2.11.0 CPU (Apple M4 Max)
YOLOv8n-cls summary (fused): 30 layers, 1,446,409 parameters, 0 gradients, 3.3 GFLOPs
train: /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_baseline/train... found 3326 images in 9 classes ✅ 
val: /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_baseline/val... found 713 images in 9 classes ✅ 
test: /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_baseline/test... found 713 images in 9 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 616.6±203.3 MB/s, size: 161.4 KB)
test: Scanning /Users/juliareinhart/Documents/CSCI611AppliedMachineLearning/BranchJuliaFinalProject/waste-classification-ml/outputs_for_YOLO_classification_baseline/test